In [1]:

import warnings
warnings.filterwarnings("ignore")
import sys
import os
from arch import arch_model
import numpy as np 
import pandas as pd
import torch
import math
from statsmodels.tsa.stattools import adfuller
import torch
import torch.nn as nn
from sklearn.metrics import mean_squared_error, r2_score
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from scipy.stats import norm
from scipy import stats
import scipy.stats as scipy_stats
import plotly.graph_objects as go

In [30]:
import importlib

## Market Return

The market return is calculated as the **market-cap weighted average** of individual stock returns:

$$R_t^{\text{market}} = \sum_{i=1}^{N} w_{i,t} \cdot r_{i,t}$$

where:
- $R_t^{\text{market}}$ = Market return at time $t$
- $N$ = Total number of stocks in the index
- $w_{i,t}$ = Weight of stock $i$ at time $t$
- $r_{i,t}$ = Return of stock $i$ at time $t$

The weight of each stock is based on its **market capitalization**:

$$w_{i,t} = \frac{\text{MC}_{i,t}}{\sum_{j=1}^{N} \text{MC}_{j,t}}$$

where:
- $\text{MC}_{i,t}$ = Market capitalization of stock $i$ at time $t$
- $\sum_{j=1}^{N} \text{MC}_{j,t}$ = Total market capitalization of all stocks

**Key Properties:**
- $\sum_{i=1}^{N} w_{i,t} = 1$ (weights sum to 100%)
- $w_{i,t} \geq 0$ for all $i, t$ (non-negative weights)
- Larger companies have higher influence on market return

In [2]:
path = "../dataset/VN30_dataset_from_2019.csv"
df = pd.read_csv(path)

df["time"] = pd.to_datetime(df["time"], format="mixed", dayfirst=True, errors="coerce")


df_2020 = df[df["time"].dt.year >= 2020]


stocks_returns = (
    df_2020
    .sort_values("time")
    .groupby("time")
    .apply(lambda x: np.average(
        x["return_1d"],
        weights=x["Market Capital (Bn VND)"] / x["Market Capital (Bn VND)"].sum()
    ))
    .dropna()
)

stocks_returns = stocks_returns.to_frame(name="stocks_return") 
stocks_returns["stocks_return"] = stocks_returns["stocks_return"] * 100





### Distribution of Return Series

The distribution of the return series was examined by fitting both a Student’s t-distribution and a normal distribution. The Kolmogorov–Smirnov test results indicate that the Student’s t-distribution provides a good fit to the data (p-value = 0.674), while the normal distribution is strongly rejected (p-value ≈ 0). 

These findings suggest that the return series exhibits heavy-tailed behavior, a common stylized fact in financial markets. Therefore, the Student’s t-distribution is more appropriate for modeling the return distribution in this dataset.

In [8]:
def distribution_analysis(data, name):
    returns = data.dropna().values
    
    nu, loc, scale = stats.t.fit(returns)
    ks_stat, ks_p = stats.kstest(returns, "t", args=(nu, loc, scale))
    
    norm_loc, norm_scale = stats.norm.fit(returns)
    ks_stat_norm, ks_p_norm = stats.kstest(returns, "norm", args=(norm_loc, norm_scale))
    return nu, loc, scale, ks_stat, ks_p, ks_stat_norm, ks_p_norm

datasets = {}

datasets['Stock returns'] = stocks_returns["stocks_return"]

df_vn30 = pd.read_csv("../dataset/VN30_INDEX.csv")
df_vn30["time"] = pd.to_datetime(df_vn30["time"], format="mixed", dayfirst=True, errors="coerce")
df_vn30_2020 = df_vn30[df_vn30["time"].dt.year >= 2020].copy()
vn30_returns = df_vn30_2020['return_1d_vn30'] * 100
datasets['VN30 Index'] = vn30_returns

df_vn = pd.read_csv("../dataset/VN_INDEX.csv")  
df_vn["time"] = pd.to_datetime(df_vn["time"], format="mixed", dayfirst=True, errors="coerce")
df_vn_2020 = df_vn[df_vn["time"].dt.year >= 2020].copy()
vn_returns = df_vn_2020['return_1d_vnindex'] * 100
datasets['VN Index'] = vn_returns

for file in ['DAX_40.csv', 'EuroNext_100.csv', 'IBEX_35.csv', 'KOSPI.csv', 'SMI.csv', 'snp500.csv', 'Topix_500.csv']:
    df_temp = pd.read_csv(f"../dataset/{file}")
    df_temp["time"] = pd.to_datetime(df_temp["time"], format="mixed", dayfirst=True, errors="coerce")
    df_temp_2020 = df_temp[df_temp["time"].dt.year >= 2020].copy()
    returns = np.log(df_temp_2020['close'] / df_temp_2020['close'].shift(1)).fillna(0) * 100
    datasets[file.replace('.csv', '')] = returns

summary_results = []
for name, data in datasets.items():
    nu, loc, scale, ks_t, p_t, ks_n, p_n = distribution_analysis(data, name)
    summary_results.append({
        'Dataset': name,
        'Nu': f"{nu:.4f}",
        'Location': f"{loc:.4f}",
        'Scale': f"{scale:.4f}",
        'KS_StudentT': f"{ks_t:.4f}",
        'P_StudentT': f"{p_t:.4f}",
        'KS_Normal': f"{ks_n:.4f}",
        'P_Normal': f"{p_n:.4f}",
        'Best_Fit': 'Student-t' if p_t > p_n else 'Normal'
    })

summary_df = pd.DataFrame(summary_results)
print(f"\n{'='*80}")
print("DISTRIBUTION ANALYSIS SUMMARY")
print(f"{'='*80}")
print(summary_df.to_string(index=False))
print(f"{'='*80}")


DISTRIBUTION ANALYSIS SUMMARY
      Dataset     Nu Location  Scale KS_StudentT P_StudentT KS_Normal P_Normal  Best_Fit
Stock returns 2.5164   0.1604 0.7706      0.0186     0.6738    0.1050   0.0000 Student-t
   VN30 Index 2.7537   0.1423 0.8766      0.1835     0.0000    0.2047   0.0000 Student-t
     VN Index 2.7542   0.1496 0.8152      0.1920     0.0000    0.2228   0.0000 Student-t
       DAX_40 2.9032   0.0893 0.7640      0.0155     0.8490    0.0957   0.0000 Student-t
 EuroNext_100 2.9521   0.0859 0.7003      0.0172     0.7479    0.0916   0.0000 Student-t
      IBEX_35 3.5449   0.0954 0.8219      0.0228     0.3926    0.0832   0.0000 Student-t
        KOSPI 4.3792   0.0838 0.9415      0.0175     0.7478    0.0537   0.0004 Student-t
          SMI 3.3731   0.0524 0.6264      0.0151     0.8780    0.0774   0.0000 Student-t
       snp500 2.8404   0.1010 0.7614      0.0156     0.8522    0.0946   0.0000 Student-t
    Topix_500 4.1379   0.0787 0.8586      0.0185     0.6875    0.0651   0.0000 

### Kolmogorov–Smirnov Test

The Kolmogorov–Smirnov (KS) test is a non-parametric statistical test used to determine whether two samples follow the same probability distribution, or whether a sample follows a specific theoretical distribution.

The test compares the **empirical cumulative distribution functions (ECDFs)** of two datasets and calculates the maximum absolute difference between them:

D = sup |F₁(x) − F₂(x)|

where:

- F₁(x) and F₂(x) are the cumulative distribution functions of the two samples  
- D is the KS statistic, representing the largest distance between the two distributions  

A **p-value** is then computed to evaluate the statistical significance of this difference.

- A **high p-value** indicates that the null hypothesis cannot be rejected, meaning the two samples likely come from the same distribution.  
- A **low p-value** suggests that the distributions are significantly different.

In this study, the KS test is used to ensure that the **training, validation, and testing datasets have similar return distributions**, which helps maintain consistency and reliability in model evaluation.

### Data Splitting Method

The dataset is divided into three subsets: training, validation, and testing. Instead of using a fixed ratio, the split points are determined by searching for the partition that maximizes the similarity of return distributions across the three subsets.

To achieve this, the Kolmogorov–Smirnov (KS) test is applied to measure the distributional similarity between the training–validation, training–test, and validation–test samples. For each possible split within predefined ranges, the average KS p-value across these three comparisons is computed. The split that yields the highest average p-value is selected as the optimal partition.

This approach ensures that the training, validation, and test sets share similar statistical distributions, thereby reducing the risk of distributional bias and improving the reliability of model evaluation.

In [9]:
def ks_optimal_split(series):

    returns = series.values
    n = len(returns)

    best_score = -1
    best_split = None

    for i in range(int(n*0.5), int(n*0.7)):
        for j in range(i + int(n*0.1), int(n*0.9)):

            train = returns[:i]
            val   = returns[i:j]
            test  = returns[j:]

            ks_tv = stats.ks_2samp(train, val).pvalue
            ks_tt = stats.ks_2samp(train, test).pvalue
            ks_vt = stats.ks_2samp(val, test).pvalue

            score = (ks_tv + ks_tt + ks_vt) / 3

            if score > best_score:
                best_score = score
                best_split = (i, j)

    i, j = best_split

    train = returns[:i]
    val   = returns[i:j]
    test  = returns[j:]

    return train, val, test, i, j, best_score



#train, val, test, i, j, score = ks_optimal_split(stocks_returns["stocks_return"])

In [10]:
results = []
for name, data in datasets.items():
    train, val, test, i, j, score = ks_optimal_split(data.dropna())
    results.append({
        'Dataset': name,
        'Train': len(train),
        'Val': len(val),
        'Test': len(test),
        'KSScore': score
,    })
summary_split = pd.DataFrame(results)
print("\n" + "="*80)
print("KS-OPTIMAL DATA SPLIT SUMMARY")
print("="*80)
print(summary_split.to_string(index=False))
print("="*80)


KS-OPTIMAL DATA SPLIT SUMMARY
      Dataset  Train  Val  Test  KSScore
Stock returns    946  184   368 0.573188
   VN30 Index    999  189   309 0.166747
     VN Index    988  149   361 0.215165
       DAX_40   1066  276   186 0.673010
 EuroNext_100    769  154   615 0.477094
      IBEX_35    962  159   421 0.375380
        KOSPI    981  162   332 0.500754
          SMI   1055  168   287 0.598059
       snp500   1054  229   225 0.362910
    Topix_500    908  175   388 0.827109


In [15]:
split_df = summary_split[["Dataset", "Train", "Val", "Test"]].set_index("Dataset")
print("\n" + "="*80)
print("KS SPLIT LENGTHS (FOR TRAINING)")
print("="*80)
print(split_df)


KS SPLIT LENGTHS (FOR TRAINING)
               Train  Val  Test
Dataset                        
Stock returns    946  184   368
VN30 Index       999  189   309
VN Index         988  149   361
DAX_40          1066  276   186
EuroNext_100     769  154   615
IBEX_35          962  159   421
KOSPI            981  162   332
SMI             1055  168   287
snp500          1054  229   225
Topix_500        908  175   388


## LSTM-GARCH Model Architecture

The LSTM-GARCH hybrid model integrates traditional econometric GARCH volatility modeling with Long Short-Term Memory (LSTM) networks to capture both classical volatility clustering patterns and complex nonlinear dependencies in financial time series.

### Model Framework

The architecture combines two main components: an enhanced GARCH foundation that models volatility clustering and leveraged effects, and an LSTM memory mechanism that captures nonlinear temporal patterns. The integration allows the model to adaptively adjust volatility forecasts based on learned market dynamics.

### Enhanced GARCH Component with HAR Features

The base volatility equation extends the classical GARCH(1,1) formulation with Heterogeneous Autoregressive (HAR) components:

$$\text{base\_var}_t = \omega + \alpha \varepsilon_{t-1}^2 + \beta \sigma_{t-1}^2 + \lambda \varepsilon_{t-1}^2 \mathbf{I}(\varepsilon_{t-1} < 0) + \phi_1 RV_{1,t-1} + \phi_5 RV_{5,t-1} + \phi_{20} RV_{20,t-1}$$

where:
- $\omega$ represents the unconditional variance
- $\alpha$ and $\beta$ are standard ARCH and GARCH coefficients  
- $\lambda$ captures leverage effects
- $RV_{1,t} = \varepsilon_{t-1}^2$, $RV_{5,t}$ = 5-day average of squared returns, $RV_{20,t}$ = 20-day average of squared returns
- $\phi_1, \phi_5, \phi_{20}$ are HAR coefficients with normalized constraints

### LSTM Memory Mechanism

The LSTM component processes standardized inputs and applies element-wise corrections:

**Input Features:**
- Standardized shock: $s_t = \frac{\varepsilon_{t-1}}{\sqrt{\sigma_{t-1}^2 + \epsilon}}$
- Log variance: $v_t = \log(\text{base\_var}_t + \epsilon)$

**LSTM Gates (Element-wise Operations):**
$$f_t = \sigma(W_f \odot s_t + U_f \odot v_t + b_f)$$
$$i_t = \sigma(W_i \odot s_t + U_i \odot v_t + b_i)$$
$$\tilde{C}_t = \tanh(W_c \odot s_t + U_c \odot v_t + b_c)$$

where $\odot$ denotes element-wise multiplication.

**Cell State Evolution:**
$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$

**Output Correction:**
$$\text{correction}_t = \tanh(C_t^T v)$$

### Hybrid Integration

The final volatility specification combines GARCH dynamics with multiplicative LSTM adjustments:

$$\sigma_t^2 = \text{base\_var}_t \times (1 + 0.05 \times \tanh(\text{correction}_t))$$

### Statistical Distribution

The model assumes returns follow a Student-t distribution with learnable degrees of freedom:

$$r_t | \mathcal{F}_{t-1} \sim \text{Student-t}(\nu, 0, \sigma_t^2)$$

where $\nu \geq 4$ is constrained to ensure finite fourth moments.

### Optimization Objective

The model is optimized using a composite loss function:

$$\mathcal{L} = \underbrace{\mathbb{E}[\log(\sigma_t^2) + \frac{\varepsilon_t^2}{\sigma_t^2}]}_{\text{QLIKE}} + \underbrace{0.05 \times \mathbb{E}[\text{ReLU}(\varepsilon_t^2 - \sigma_t^2)^2]}_{\text{Spike Penalty}}$$

This loss function combines quasi-maximum likelihood estimation with regularization to prevent extreme volatility spikes.

In [16]:

class LSTMGARCH(nn.Module):
    def __init__(self):
        super().__init__()

        self.hidden_dim = 16

        self.raw_omega = nn.Parameter(torch.tensor(-5.0))
        self.raw_alpha = nn.Parameter(torch.tensor(-2.0))
        self.raw_beta  = nn.Parameter(torch.tensor(0.0))
        self.raw_lambda = nn.Parameter(torch.tensor(-2.0))

        self.raw_phi1 = nn.Parameter(torch.tensor(-1.0))
        self.raw_phi5 = nn.Parameter(torch.tensor(-1.0))
        self.raw_phi20 = nn.Parameter(torch.tensor(-1.0))

        self.raw_gamma = nn.Parameter(torch.tensor(-2.0))

        self.Wf = nn.Parameter(torch.randn(self.hidden_dim))
        self.Uf = nn.Parameter(torch.randn(self.hidden_dim))
        self.bf = nn.Parameter(torch.zeros(self.hidden_dim))

        self.Wi = nn.Parameter(torch.randn(self.hidden_dim))
        self.Ui = nn.Parameter(torch.randn(self.hidden_dim))
        self.bi = nn.Parameter(torch.zeros(self.hidden_dim))

        self.Wc = nn.Parameter(torch.randn(self.hidden_dim))
        self.Uc = nn.Parameter(torch.randn(self.hidden_dim))
        self.bc = nn.Parameter(torch.zeros(self.hidden_dim))

        self.v = nn.Parameter(torch.randn(self.hidden_dim))

        self.w = nn.Parameter(torch.tensor(0.0))

        self.raw_nu = nn.Parameter(torch.tensor(4.0))
    def garch_params(self):

        omega = F.softplus(self.raw_omega)

        alpha = torch.sigmoid(self.raw_alpha)
        beta  = torch.sigmoid(self.raw_beta)

        scale = alpha + beta + 1e-6
        alpha = alpha / scale * 0.95
        beta  = beta  / scale * 0.95

        lambda_ = torch.sigmoid(self.raw_lambda)

        phi1 = torch.sigmoid(self.raw_phi1)
        phi5 = torch.sigmoid(self.raw_phi5)
        phi20 = torch.sigmoid(self.raw_phi20)

        phi_sum = phi1 + phi5 + phi20 + 1e-6
        phi1 = phi1 / phi_sum * 0.6
        phi5 = phi5 / phi_sum * 0.3
        phi20 = phi20 / phi_sum * 0.1

        gamma = torch.sigmoid(self.raw_gamma)

        return omega, alpha, beta, lambda_, phi1, phi5, phi20, gamma

    def student_nu(self):
        return torch.clamp(F.softplus(self.raw_nu) + 2, min=4)

    def forward(self, returns):

        omega, alpha, beta, lambda_, phi1, phi5, phi20, gamma = self.garch_params()
        nu = self.student_nu()

        batch_size, T = returns.shape

        sigma2_list = []

        c_t = torch.zeros(batch_size, self.hidden_dim, device=returns.device)

        sigma2_t = returns[:,0]**2 + 1e-6
        sigma2_list.append(sigma2_t)

        squared_returns_history = []
        squared_returns_history.append(returns[:,0]**2)

        for t in range(1, T):

            eps_prev = returns[:, t-1]
            shock = eps_prev / torch.sqrt(sigma2_t + 1e-8)

            neg = (eps_prev < 0).float()
            leverage = lambda_ * eps_prev**2 * neg

            RV1 = eps_prev**2
            
            if len(squared_returns_history) >= 5:
                RV5 = torch.stack(squared_returns_history[-5:], dim=1).mean(dim=1)
            else:
                RV5 = torch.stack(squared_returns_history, dim=1).mean(dim=1)
            
            if len(squared_returns_history) >= 20:
                RV20 = torch.stack(squared_returns_history[-20:], dim=1).mean(dim=1)
            else:
                RV20 = torch.stack(squared_returns_history, dim=1).mean(dim=1)

            base_var = omega + alpha*eps_prev**2 + beta*sigma2_t + leverage + phi1*RV1 + phi5*RV5 + phi20*RV20

            log_sigma = torch.log(base_var + 1e-8)

            shock = shock.unsqueeze(1)
            log_sigma = log_sigma.unsqueeze(1)

            f_t = torch.sigmoid(self.Wf * shock + self.Uf * log_sigma + self.bf)
            i_t = torch.sigmoid(self.Wi * shock + self.Ui * log_sigma + self.bi)
            c_hat = torch.tanh(self.Wc * shock + self.Uc * log_sigma + self.bc)

            c_t = f_t * c_t + i_t * c_hat

            correction = torch.tanh((c_t * self.v).sum(dim=1))
            
            sigma2_t = base_var * (1 + 0.05 * torch.tanh(correction))

            sigma2_list.append(sigma2_t)
            squared_returns_history.append(eps_prev**2)

        sigma2 = torch.stack(sigma2_list, dim=1)

        eps = returns

        qlike = torch.log(sigma2) + eps**2 / sigma2

        spike_penalty = torch.relu(eps**2 - sigma2)**2



        loss = qlike.mean() + 0.05*spike_penalty.mean()

        return loss, sigma2

Training the LSTM - GARCH models

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seq_len = 60

def create_sequences(data, seq_len):
    xs = []
    for i in range(len(data) - seq_len):
        xs.append(data[i:i+seq_len])
    return np.array(xs, dtype=np.float32)

def compute_var_student_t(volatility, confidence_level, nu):
    t_alpha = scipy_stats.t.ppf(confidence_level, df=nu)
    return t_alpha * volatility * np.sqrt((nu - 2) / nu)

def compute_metrics(realized_vol, predicted_vol, realized_var, predicted_var):
    mse = np.mean((realized_vol - predicted_vol) ** 2)
    qlike = np.mean(np.log(predicted_var) + realized_var / predicted_var)
    return mse, qlike

def kupiec_test(violations, confidence_level=0.95):
    n = len(violations)
    n_viol = violations.sum()
    expected = 1 - confidence_level
    if n_viol == 0 or n_viol == n:
        return np.nan, np.nan
    obs = n_viol / n
    lr = -2 * ((n - n_viol) * np.log(1 - expected) + n_viol * np.log(expected) - (n - n_viol) * np.log(1 - obs) - n_viol * np.log(obs))
    p = 1 - scipy_stats.chi2.cdf(lr, 1)
    return lr, p

def evaluate_model(returns, volatility, confidence_level, nu):
    var_threshold = compute_var_student_t(volatility, confidence_level, nu)
    violations = returns < -var_threshold
    viol_rate = violations.mean()
    lr, p = kupiec_test(violations, confidence_level)
    return viol_rate, lr, p

def get_ml_predictions(model, data_loader):
    model.eval()
    preds = []
    with torch.no_grad():
        for batch in data_loader:
            _, var_seq = model(batch)
            preds.extend(var_seq[:, -1].sqrt().cpu().numpy())
    return np.array(preds)

def train_lstm_garch_for_series(name, series, splits, epochs=100):
    if name not in splits.index:
        return None
    s = series.dropna().values.astype(np.float32)
    train_len = int(splits.loc[name, "Train"])
    val_len = int(splits.loc[name, "Val"])
    test_len = int(splits.loc[name, "Test"])
    if train_len + val_len + test_len > len(s) or train_len <= seq_len or test_len <= seq_len:
        return None
    train = s[:train_len]
    val = s[train_len:train_len + val_len]
    test = s[train_len + val_len:train_len + val_len + test_len]
    train_seq = create_sequences(train, seq_len)
    test_seq = create_sequences(test, seq_len)
    train_tensor = torch.tensor(train_seq, dtype=torch.float32, device=device)
    test_tensor = torch.tensor(test_seq, dtype=torch.float32, device=device)
    train_loader = DataLoader(train_tensor, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_tensor, batch_size=64, shuffle=False)
    model = LSTMGARCH().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=0.001)
    for _ in range(epochs):
        model.train()
        for batch in train_loader:
            opt.zero_grad()
            loss, _ = model(batch)
            loss.backward()
            opt.step()
    ml_vol = get_ml_predictions(model, test_loader)
    returns_test = test
    returns_eval = returns_test[seq_len:]
    m = min(len(ml_vol), len(returns_eval))
    ml_vol = ml_vol[:m]
    returns_eval = returns_eval[:m]
    realized_vol = np.abs(returns_eval)
    realized_var = returns_eval ** 2
    ml_var = ml_vol ** 2
    mse, qlike = compute_metrics(realized_vol, ml_vol, realized_var, ml_var)
    nu, _, _ = stats.t.fit(series.dropna().values)
    viol_rate, kupiec_lr, kupiec_p = evaluate_model(returns_eval, ml_vol, 0.95, nu)
    return {
        "Dataset": name,
        "MSE": f"{mse:.6f}",
        "QLIKE": f"{qlike:.6f}",
        "Violation_Rate": f"{viol_rate:.4f}",
        "Kupiec_LR": f"{kupiec_lr:.4f}" if not np.isnan(kupiec_lr) else "N/A",
        "Kupiec_p": f"{kupiec_p:.4f}" if not np.isnan(kupiec_p) else "N/A",
    }

results = []
for name, series in datasets.items():
    r = train_lstm_garch_for_series(name, series, split_df, epochs=100)
    if r is not None:
        results.append(r)
results_df = pd.DataFrame(results)
print("\n" + "="*80)
print("LSTM-GARCH CROSS-DATASET PERFORMANCE SUMMARY")
print("="*80)
print(results_df.to_string(index=False))
print("="*80)


LSTM-GARCH CROSS-DATASET PERFORMANCE SUMMARY
      Dataset      MSE    QLIKE Violation_Rate Kupiec_LR Kupiec_p
Stock returns 2.281838 1.439523         0.0487    0.0110   0.9164
   VN30 Index 2.921405 1.878066         0.0361    1.1091   0.2923
     VN Index 2.408644 1.576787         0.0465    0.0789   0.7788
       DAX_40 1.361153 1.176733         0.0159    4.1630   0.0413
 EuroNext_100 1.425896 0.879146         0.0252    8.6985   0.0032
      IBEX_35 1.939084 1.286180         0.0166   11.3019   0.0008
        KOSPI 1.622055 1.711673         0.0625    0.8318   0.3617
          SMI 0.944941 0.697401         0.0485    0.0115   0.9147
       snp500 0.704222 0.655588         0.0242    2.8229   0.0929
    Topix_500 1.376621 1.307481         0.0366    1.3648   0.2427


Other models

In [ ]:


# ==============================================================================
# 6. Run the pipeline
# ==============================================================================
# Ensure all modules are fresh
import ultility.data_loader
import ultility.metrics
import ultility.models_garch
import ultility.models_transformer
import train_pipeline

importlib.reload(ultility.data_loader)
importlib.reload(ultility.metrics)
importlib.reload(ultility.models_garch)
importlib.reload(ultility.models_transformer)
importlib.reload(train_pipeline)


print("Starting the benchmarking pipeline...")
#results_df = train_pipeline.run_benchmark(datasets, split_df, seq_len=60)

print("\\n" + "="*80)
print("BASELINE MODELS BENCHMARKING RESULTS")
print("="*80)
display(results_df)
print("="*80)


100%|██████████| 100/100 [10:23<00:00,  6.23s/it]


,Dataset,Model,MSE,QLIKE,Violation_Rate,Kupiec_LR,Kupiec_p
0,DAX_40,GARCH,3.875603,0.808236,0.053763,0.054191,0.815925
1,DAX_40,GJR-GARCH,3.916693,0.808976,0.075269,2.179340,0.139874
2,DAX_40,Transformer,1.114349,0.640157,0.103175,5.816613,0.015875
3,EuroNext_100,GARCH,2.640505,0.431952,0.086179,14.064917,0.000177
4,EuroNext_100,GJR-GARCH,3.319717,0.389140,0.086179,14.064917,0.000177
5,EuroNext_100,Transformer,3.407285,0.490130,0.099099,22.183987,0.000002
6,IBEX_35,GARCH,6.834103,0.824298,0.076010,5.208030,0.022483
7,IBEX_35,GJR-GARCH,5.609360,0.821570,0.066508,2.198498,0.138145
8,IBEX_35,Transformer,7.354217,1.058396,0.094183,11.912191,0.000558
9,KOSPI,GARCH,25.002422,1.674093,0.060241,0.689967,0.406175


In [ ]:

import ultility.data_loader
import ultility.metrics
import ultility.models_garch
import ultility.models_transformer
import train_pipeline

importlib.reload(ultility.data_loader)
importlib.reload(ultility.metrics)
importlib.reload(ultility.models_garch)
importlib.reload(ultility.models_transformer)
importlib.reload(train_pipeline)


print("Starting the benchmarking pipeline...")
results_df = train_pipeline.run_benchmark(datasets, split_df, seq_len=60)

print("\\n" + "="*80)
print("BASELINE MODELS BENCHMARKING RESULTS")
print("="*80)
display(results_df)
print("="*80)


Starting the benchmarking pipeline...
\n================================================================================
BASELINE MODELS BENCHMARKING RESULTS


,Dataset,Model,MSE,QLIKE,Violation_Rate,Kupiec_LR,Kupiec_p
0,DAX_40,GARCH,3.875603,0.808236,0.053763,0.054191,0.815925
1,DAX_40,GJR-GARCH,3.916693,0.808976,0.075269,2.179340,0.139874
2,DAX_40,Transformer,1.114349,0.640157,0.103175,5.816613,0.015875
3,EuroNext_100,GARCH,2.640505,0.431952,0.086179,14.064917,0.000177
4,EuroNext_100,GJR-GARCH,3.319717,0.389140,0.086179,14.064917,0.000177
5,EuroNext_100,Transformer,3.407285,0.490130,0.099099,22.183987,0.000002
6,IBEX_35,GARCH,6.834103,0.824298,0.076010,5.208030,0.022483
7,IBEX_35,GJR-GARCH,5.609360,0.821570,0.066508,2.198498,0.138145
8,IBEX_35,Transformer,7.354217,1.058396,0.094183,11.912191,0.000558
9,KOSPI,GARCH,25.002422,1.674093,0.060241,0.689967,0.406175
